# Final model: RT-DETR-L - walkthrough and reproduction

This notebook shows the full pipeline of the final model step by step on the mini dataset (CPU, ~5 min), then how the same code is run on Kaggle GPU for the real result (val mAP 0.252, LB 0.340 public / 0.270 private).

Everything is imported from the repo; nothing is reimplemented here.

## 1. Code
Clone the repo and install the pinned environment (`code/requirements.txt` = CPU torch + ultralytics 8.4.214 + pycocotools). On Kaggle only `ultralytics==8.4.214 pycocotools` are installed on top of the preinstalled CUDA torch.

In [ ]:
!git clone --depth 1 https://github.com/BodiyaKol/ucu-ai-hm-2-object-detection_Kolodchak.git repo
%cd repo/code
!pip install -q -r requirements.txt

## 2. Data
The mini dataset (2 clips x 20 frames per split, exact competition layout: `train/`, `val/`, `test/`, `test_images.csv`) is the one every script was tested on. It is shared here:
https://drive.google.com/drive/folders/1mfM_7MvppELbCmmQBzC53E34yruDev4r (folder `data_for_notebook/mini`). The cell below downloads it into `code/data/mini/`. On Kaggle the full competition data is found automatically by `src.config.get_data_root`.

In [ ]:
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/1mfM_7MvppELbCmmQBzC53E34yruDev4r" -O data_dl --remaining-ok
!mkdir -p data && cp -r data_dl/data_for_notebook/mini data/ && rm -rf data_dl
import sys; sys.path.insert(0, 'scripts')
from src.config import load_config, get_data_root
cfg = load_config('configs/protocol.yaml')
root = get_data_root(cfg)
print('data root:', root)
!ls {root}

## 3. Preparation (`scripts/prepare_data.py`, M1.5)
Every 10th train frame, ignore regions painted gray (114), boxes covered >50% by ignore dropped, YOLO labels + `train.txt`, COCO json for val with ignore regions as `iscrowd=1`. Output: `outputs/prepared/`. On Kaggle this is the `hw2-01-prepare` kernel, attached to training kernels as `kernel_sources`.

In [ ]:
!python scripts/prepare_data.py
!cat outputs/prepared/prepare_stats.json

## 4. Config
Shared protocol (`configs/protocol.yaml`: seed 42, frame_step 10, imgsz 1024, 500 detections, score 0.001) + model config (`configs/rtdetr/base.yaml`: `rtdetr-l.pt`, batch 4, time cap 0.75 h). `mini:` block overrides for the CPU dry-run.

In [ ]:
!cat configs/protocol.yaml
!cat configs/rtdetr/base.yaml

## 5. Training + evaluation + submission (`scripts/train_yolo.py`)
One script, same for YOLO and RT-DETR (class picked from the weights name):
1. `write_data_yaml` -> Ultralytics data.yaml (train = prepared, val = original val images)
2. `RTDETR(...).train(...)` with the config values; `time` caps the run
3. `predict`: images passed as a .txt list, batch 1 (a Python list would be decoded into RAM as one batch -> OOM)
4. `coco_evaluate` (`src/eval/coco_eval.py`): pycocotools, top-500, ignore regions as crowd, per-class AP
5. `predictions_to_csv` + `check_submission` (`src/submission.py`): all 3811 test rows, top-500 by score

Mini run below: 1 epoch, 320 px, CPU.

In [ ]:
!python scripts/train_yolo.py --exp nb_mini --mini --config configs/rtdetr/base.yaml

In [ ]:
import json
m = json.load(open('outputs/nb_mini/metrics.json'))
print(json.dumps({k: m[k] for k in ['exp', 'weights', 'imgsz', 'epochs', 'val', 'submission']}, indent=2))
!head -c 300 outputs/nb_mini/submission.csv

## 6. Real run on Kaggle GPU
The same script is launched by the kernel folder `notebooks/06_rtdetr/` (`run.py` clones the repo and runs training then inference in two processes; `kernel-metadata.json` attaches the competition data and the `hw2-01-prepare` output). From `code/`:

```bash
kaggle kernels push -p notebooks/06_rtdetr --accelerator NvidiaTeslaT4
kaggle kernels status bohdankolodchak/hw2-06-rtdetr
kaggle kernels output bohdankolodchak/hw2-06-rtdetr -p outputs/06 --file-pattern "metrics.json|submission.csv"
kaggle competitions submit -c multiple-object-video-detection-hw-2-ucu-ai-2026 -f outputs/06/m6_rtdetr/submission.csv -m "RT-DETR-L 1024"
```

Test inference from saved weights (`notebooks/08b_late_rtdetr/`) attaches the finished kernel as `kernel_sources` and runs `train_yolo.py --weights .../last.pt`.

Seeds: 42 everywhere (`set_seed`), Ultralytics `deterministic=True`. Results of all runs: `code/results/*.json`, `code/experiments.md`.